In [1]:
import sys
sys.path.append("..")
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datasets import load_from_disk
from tqdm import tqdm

from src.data import PROJECT_ROOT
from src.llm_upgrade import wrap_for_transformer_lens
from src.sae_test import TopKSAE
from src.rule_extraction import rank_features_by_logic
from src.interventions import run_ablation, run_patching, compute_logit_shifts, get_flipped_texts

In [2]:
EXP_ID = "exp6-1"
MODEL_NAME = "gpt2-large"
VARIANT = "depth-0"
ADAPTER_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-11000")
PROBING_JSON = PROJECT_ROOT / "results/probing/probe_gpt2-large(qLoRA)_depth-0_resid_post_last.json"
with open(PROBING_JSON, "r", encoding="utf-8") as f:
    probing_data = json.load(f)
BEST_LAYER = probing_data["summary"]["best_layer"]
SAE_DIR = PROJECT_ROOT / f"results/sae/{EXP_ID}_centered/layer_{BEST_LAYER}"
SAE_PATH = str(SAE_DIR / "sae_final.pt")

In [3]:
BASELINE_DIR = PROJECT_ROOT / "results/baselines" / EXP_ID
CAUSAL_DIR = BASELINE_DIR / "causal_centered"
CAUSAL_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
N_CAUSAL_SAMPLES = 500
BATCH_SIZE = 32
TOP_K_LATENTS = 20

In [5]:
torch.cuda.empty_cache()

# Загрузка модели, датасета, SAE

In [6]:
hooked_model, tokenizer = wrap_for_transformer_lens(MODEL_NAME, ADAPTER_PATH, device="cuda")
hooked_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Loaded pretrained model gpt2-large into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_re

In [7]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
full_test = load_from_disk(str(cache_path))["test"]

In [8]:
checkpoint = torch.load(SAE_PATH, map_location="cuda", weights_only=False)
K_SAE_CFG = checkpoint['config']['k']
D_SAE = checkpoint['config']['d_sae']
D_MODEL = checkpoint['config']['d_in']
state_dict = checkpoint["model_state_dict"]
mean = state_dict.get("mean", None)

In [9]:
sae = TopKSAE(d_in=D_MODEL, d_sae=D_SAE, k=K_SAE_CFG, mean=mean).to("cuda")
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

TopKSAE()

In [10]:
activations = np.load(SAE_DIR / "test_activations_formatted.npy")
labels = np.load(SAE_DIR / "test_labels_formatted.npy")
llm_preds = torch.load(BASELINE_DIR / "cache" / "test_acts_preds.pt", map_location="cpu", weights_only=False)["preds"].numpy()

In [11]:
eval_prefix = "{theory} {assertion} The assertion is"
def format_text(item):
    if "theory" in item and "assertion" in item:
        return eval_prefix.format(theory=item["theory"], assertion=item["assertion"])
    return item.get("text", "")

dataset_texts = [format_text(full_test[i]) for i in range(len(labels))]

In [12]:
rng = np.random.default_rng(42)
idx_true = np.where(labels == 1)[0]
idx_false = np.where(labels == 0)[0]
selected_idx = np.concatenate([
    rng.choice(idx_true, size=min(N_CAUSAL_SAMPLES//2, len(idx_true)), replace=False),
    rng.choice(idx_false, size=min(N_CAUSAL_SAMPLES//2, len(idx_false)), replace=False)
])
rng.shuffle(selected_idx)

In [13]:
texts_causal = [dataset_texts[i] for i in selected_idx]
llm_causal = llm_preds[selected_idx]
labels_causal = labels[selected_idx]
prompt_lens = [len(tokenizer.encode(txt, add_special_tokens=False)) for txt in texts_causal]
print(f"Выборка: {len(texts_causal)} примеров (баланс: {np.mean(labels_causal):.2f})")

Выборка: 500 примеров (баланс: 0.50)


In [14]:
true_ids = tokenizer.encode("True", add_special_tokens=False)
false_ids = tokenizer.encode("False", add_special_tokens=False)
TRUE_ID = true_ids[-1] if true_ids else 0
FALSE_ID = false_ids[-1] if false_ids else 0

In [15]:
HOOK_NAME = f"blocks.{BEST_LAYER}.hook_resid_post"

In [16]:
top_feat, corr_label, corr_logic, combined = rank_features_by_logic(activations[selected_idx], llm_causal, texts_causal)
candidate_latents = top_feat[:TOP_K_LATENTS].tolist()

# 1. Абляция: ранжирование латентов по каузальному эффекту

In [17]:
df_ablation = run_ablation(hooked_model, sae, texts_causal, llm_causal, labels_causal, prompt_lens,
                           candidate_latents, HOOK_NAME, TRUE_ID, FALSE_ID, batch_size=BATCH_SIZE, device="cuda")

Ablation: 100%|██████████| 20/20 [19:52<00:00, 59.60s/it]


In [18]:
df_ablation = df_ablation.sort_values("delta_fidelity", ascending=False).reset_index(drop=True)
df_ablation.to_csv(CAUSAL_DIR / "ablation_results.csv", index=False)

In [19]:
df_ablation = pd.read_csv(CAUSAL_DIR / "ablation_results.csv")
df_ablation

,latent_idx,fidelity_before,fidelity_after,delta_fidelity,accuracy_before,accuracy_after,delta_accuracy,flip_rate
0,13166,1.0,1.0,0.0,0.998,0.998,0.0,0.0
1,13630,1.0,1.0,0.0,0.998,0.998,0.0,0.0
2,10569,1.0,1.0,0.0,0.998,0.998,0.0,0.0
3,994,1.0,1.0,0.0,0.998,0.998,0.0,0.0
4,13381,1.0,1.0,0.0,0.998,0.998,0.0,0.0
5,2867,1.0,1.0,0.0,0.998,0.998,0.0,0.0
6,13586,1.0,1.0,0.0,0.998,0.998,0.0,0.0
7,10877,1.0,1.0,0.0,0.998,0.998,0.0,0.0
8,15384,1.0,1.0,0.0,0.998,0.998,0.0,0.0
9,2321,1.0,1.0,0.0,0.998,0.998,0.0,0.0


# 2. Патчинг

In [20]:
df_patching = run_patching(hooked_model, sae, texts_causal, labels_causal, prompt_lens,
                           candidate_latents, HOOK_NAME, TRUE_ID, FALSE_ID, n_pairs=30, device="cuda")

Patching: 100%|██████████| 20/20 [02:50<00:00,  8.51s/it]


In [21]:
df_patching.to_csv(CAUSAL_DIR / "patching_results.csv", index=False)

In [22]:
df_patching = pd.read_csv(CAUSAL_DIR / "patching_results.csv")
display(df_patching)

,latent_idx,tested_pairs,flips,flip_rate,direction
0,13166,30,6,0.2000,False->True (sufficiency)
1,13630,30,0,0.0000,False->True (sufficiency)
2,3893,30,2,0.0667,False->True (sufficiency)
3,13836,30,2,0.0667,False->True (sufficiency)
4,12428,30,2,0.0667,False->True (sufficiency)
5,4124,30,1,0.0333,False->True (sufficiency)
6,8297,30,3,0.1000,False->True (sufficiency)
7,11319,30,3,0.1000,False->True (sufficiency)
8,2810,30,1,0.0333,False->True (sufficiency)
9,9512,30,2,0.0667,False->True (sufficiency)


# 3. Сдвиг логитов

In [23]:
# diffs_b, diffs_a = compute_logit_shifts(hooked_model, sae, texts_causal, prompt_lens, candidate_latents[0],
#                                         HOOK_NAME, TRUE_ID, FALSE_ID, batch_size=64, device="cuda")

In [24]:
# np.savez(CAUSAL_DIR / "logit_shifts.npz", diffs_b=diffs_b, diffs_a=diffs_a)

In [25]:
# data = np.load(CAUSAL_DIR / "logit_shifts.npz")
# diffs_b, diffs_a = data["diffs_b"], data["diffs_a"]
# shifts = diffs_a - diffs_b

In [26]:
# plt.figure(figsize=(7, 4))
# plt.hist(shifts, bins=50, alpha=0.7, color="#4682B4", edgecolor="k")
# plt.axvline(0, color="red", linestyle="--", linewidth=1.5)
# plt.xlabel("Сдвиг разности логитов (True - False)")
# plt.ylabel("Частота")
# plt.title(f"Распределение сдвигов логитов при абляции латента #{candidate_latents[0]}")
# plt.grid(axis="y", alpha=0.3)
# plt.tight_layout()
# plt.show()

In [27]:
# print(f"Средний сдвиг: {shifts.mean():.4f} | Медиана: {np.median(shifts):.4f} | STD: {shifts.std():.4f}")

# 4. Качественный анализ

In [28]:
# flipped_list = get_flipped_texts(hooked_model, sae, texts_causal, labels_causal, prompt_lens,
#                                  candidate_latents[0], HOOK_NAME, TRUE_ID, FALSE_ID, max_count=5, device="cuda")

In [29]:
# for ex in flipped_list:
#     gt_str = "True" if ex["gt"] == 1 else "False"
#     print(f"[GT: {gt_str}] {ex['pred_before']} -> {ex['pred_after']}")
#     print(f"  {ex['text']}\n")

In [30]:
summary = pd.DataFrame({
    "Метрика": ["delta_Fid (среднее)", "delta_Acc (среднее)", "Flip Rate (абляция)",
                "Flip Rate (патчинг)"],
    "Значение": [
        df_ablation["delta_fidelity"].mean(),
        df_ablation["delta_accuracy"].mean(),
        df_ablation["flip_rate"].mean(),
        df_patching["flip_rate"].mean()
    ]
})
display(summary)

,Метрика,Значение
0,delta_Fid (среднее),0.00000
1,delta_Acc (среднее),0.00000
2,Flip Rate (абляция),0.00000
3,Flip Rate (патчинг),0.06167
